# Part 6: ATProto Identifiers

This notebook covers the five identifier types used across ATProto: human-readable handles,
cryptographic DIDs, NSIDs, TIDs, and AT URIs.

**What you'll learn:**

- Handle and DID resolution (two-tier identity)
- NSID structure (method naming in XRPC)
- TID generation (clock-based record keys)
- AT URI parsing and resolution (`at://authority/collection/rkey`)

**Prerequisites:** Parts 1-5 (ObjC foundations and design patterns)

**Estimated Time:** 25-30 minutes

## Step 1: Handle Resolver

The handle resolver maps handles to DIDs. In a real ATProto implementation, handles are resolved via
DNS TXT records or HTTP well-known endpoints. Here we use an in-memory dictionary for simplicity.

In [ ]:
@interface HandleResolver : NSObject
@property (nonatomic, strong) NSMutableDictionary *handleToDid;
- (void)registerHandle:(NSString *)handle did:(NSString *)did;
- (NSString *)resolveHandle:(NSString *)handle;
- (BOOL)isValidHandleFormat:(NSString *)handle;
@end

@implementation HandleResolver
- (instancetype)init {
    self = [super init];
    if (self) { _handleToDid = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)registerHandle:(NSString *)handle did:(NSString *)did {
    [_handleToDid setObject:did forKey:handle];
    NSLog(@"Registered: %@ -> %@", handle, did);
}
- (NSString *)resolveHandle:(NSString *)handle {
    NSString *did = [_handleToDid objectForKey:handle];
    if (did != nil) {
        NSLog(@"Resolved %@ -> %@", handle, did);
    } else {
        NSLog(@"Handle not found: %@", handle);
    }
    return did;
}
- (BOOL)isValidHandleFormat:(NSString *)handle {
    if (handle == nil || [handle length] < 3) return NO;
    if (![handle containsString:@"."]) return NO;
    if ([handle containsString:@" "]) return NO;
    return YES;
}
@end

HandleResolver *hr = [[HandleResolver alloc] init];
[hr registerHandle:@"alice.bsky.social" did:@"did:plc:abc123"];
[hr registerHandle:@"bob.example.com" did:@"did:plc:def456"];

NSLog(@"Alice: %@", [hr resolveHandle:@"alice.bsky.social"]);
NSLog(@"Unknown: %@", [hr resolveHandle:@"unknown.bsky.social"]);
NSLog(@"Valid format: %d", [hr isValidHandleFormat:@"alice.bsky.social"]);
NSLog(@"Invalid: %d", [hr isValidHandleFormat:@"nodot"]);

## Step 2: DID Resolver

The DID resolver maps DIDs to DID documents. A DID document contains `alsoKnownAs` (associated
handles), `verificationMethod` (public keys), and `service` endpoints. In practice, DID documents
are fetched from PLC directories or web DID endpoints.

In [ ]:
@interface DIDResolver : NSObject
@property (nonatomic, strong) NSMutableDictionary *didToDocument;
- (void)registerDid:(NSString *)did document:(NSDictionary *)document;
- (NSDictionary *)resolveDid:(NSString *)did;
@end

@implementation DIDResolver
- (instancetype)init {
    self = [super init];
    if (self) { _didToDocument = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)registerDid:(NSString *)did document:(NSDictionary *)document {
    [_didToDocument setObject:document forKey:did];
    NSLog(@"Registered DID: %@", did);
}
- (NSDictionary *)resolveDid:(NSString *)did {
    NSDictionary *doc = [_didToDocument objectForKey:did];
    if (doc != nil) {
        NSLog(@"Resolved DID: %@", did);
    } else {
        NSLog(@"DID not found: %@", did);
    }
    return doc;
}
@end

DIDResolver *dr = [[DIDResolver alloc] init];
[dr registerDid:@"did:plc:abc123" document:@{
    @"alsoKnownAs": @[@"at://alice.bsky.social"],
    @"verificationMethod": @[@{@"id": @"#atproto", @"type": @"Multikey"}],
    @"service": @[@{@"id": @"#atproto_pds", @"type": @"AtprotoPersonalDataServer"}]
}];

NSDictionary *doc = [dr resolveDid:@"did:plc:abc123"];
NSLog(@"alsoKnownAs: %@", [doc objectForKey:@"alsoKnownAs"]);

## Step 3: Identity Service

The identity service chains both resolvers: handle → DID → document. This two-tier resolution
pattern is fundamental to ATProto identity.

In [ ]:
@interface IdentityService : NSObject
@property (nonatomic, strong) HandleResolver *handleResolver;
@property (nonatomic, strong) DIDResolver *didResolver;
- (NSDictionary *)resolveHandle:(NSString *)handle;
- (NSDictionary *)resolveDid:(NSString *)did;
@end

@implementation IdentityService
- (instancetype)init {
    self = [super init];
    if (self) {
        _handleResolver = [[HandleResolver alloc] init];
        _didResolver = [[DIDResolver alloc] init];
    }
    return self;
}
- (NSDictionary *)resolveHandle:(NSString *)handle {
    // Step 1: handle → DID
    NSString *did = [_handleResolver resolveHandle:handle];
    if (did == nil) return nil;
    // Step 2: DID → document
    return [_didResolver resolveDid:did];
}
- (NSDictionary *)resolveDid:(NSString *)did {
    return [_didResolver resolveDid:did];
}
@end

// Wire it up
IdentityService *idSvc = [[IdentityService alloc] init];
[idSvc.handleResolver registerHandle:@"alice.bsky.social" did:@"did:plc:abc123"];
[idSvc.didResolver registerDid:@"did:plc:abc123" document:@{
    @"alsoKnownAs": @[@"at://alice.bsky.social"],
    @"verificationMethod": @[@{@"id": @"#atproto", @"type": @"Multikey"}],
    @"service": @[@{@"id": @"#atproto_pds", @"type": @"AtprotoPersonalDataServer"}]
}];

// Resolve handle → DID → document
NSDictionary *doc = [idSvc resolveHandle:@"alice.bsky.social"];
NSLog(@"Document: %@", doc);

// Resolve DID directly
NSDictionary *doc2 = [idSvc resolveDid:@"did:plc:abc123"];
NSLog(@"Same doc: %d", [doc isEqualToDictionary:doc2]);

## Step 4: NSID — Method Names

NSIDs (Namespaced Identifiers) name XRPC methods and record collections. They follow the reverse-DNS
convention: `com.atproto.server.createSession`. Each segment is a lowercase letter sequence,
separated by dots.

In [ ]:
@interface NSID : NSObject
@property (nonatomic, strong) NSString *domain;
@property (nonatomic, strong) NSString *methodName;
- (instancetype)initWithString:(NSString *)nsid;
- (NSString *)stringValue;
- (NSString *)collectionName;
@end

@implementation NSID
- (instancetype)initWithString:(NSString *)nsid {
    self = [super init];
    if (self) {
        NSArray *parts = [nsid componentsSeparatedByString:@"."];
        if ([parts count] >= 2) {
            NSMutableArray *domainParts = [NSMutableArray array];
            for (int i = 0; i < [parts count] - 1; i++) {
                [domainParts addObject:[parts objectAtIndex:i]];
            }
            _domain = [domainParts componentsJoinedByString:@"."];
            _methodName = [parts objectAtIndex:[parts count] - 1];
        }
    }
    return self;
}
- (NSString *)stringValue {
    return [NSString stringWithFormat:@"%@.%@", _domain, _methodName];
}
- (NSString *)collectionName {
    NSArray *domainParts = [_domain componentsSeparatedByString:@"."];
    NSMutableArray *reversed = [NSMutableArray array];
    for (int i = [domainParts count] - 1; i >= 0; i--) {
        [reversed addObject:[domainParts objectAtIndex:i]];
    }
    return [NSString stringWithFormat:@"%@.%@", [reversed componentsJoinedByString:@"."], _methodName];
}
@end

NSID *nsid1 = [[NSID alloc] initWithString:@"com.atproto.server.createSession"];
NSLog(@"Domain: %@", nsid1.domain);
NSLog(@"Method: %@", nsid1.methodName);

NSID *nsid2 = [[NSID alloc] initWithString:@"app.bsky.feed.post"];
NSLog(@"Collection: %@", [nsid2 collectionName]);

## Step 5: TID — Clock-Based Record Keys

TIDs (Timestamp IDs) are base32-encoded timestamps used as record keys. They provide a monotonically
increasing, URL-safe identifier. In production, TIDs encode a 53-bit microsecond timestamp.

In [ ]:
// Simplified TID generator — uses a counter instead of real clock
@interface TIDGenerator : NSObject
@property (nonatomic, assign) int counter;
- (NSString *)nextTID;
@end

@implementation TIDGenerator
- (instancetype)init {
    self = [super init];
    if (self) { _counter = 1000; }
    return self;
}
- (NSString *)nextTID {
    _counter = _counter + 1;
    return [NSString stringWithFormat:@"3k%05xabc", _counter];
}
@end

TIDGenerator *tidGen = [[TIDGenerator alloc] init];
NSLog(@"TID 1: %@", [tidGen nextTID]);
NSLog(@"TID 2: %@", [tidGen nextTID]);
NSLog(@"TID 3: %@", [tidGen nextTID]);

## Step 6: AT URI — Record Addressing

AT URIs are the addressing scheme for ATProto records: `at://authority/collection/rkey`. The
authority is a DID or handle, the collection is an NSID, and the rkey is a TID.

In [ ]:
@interface ATURI : NSObject
@property (nonatomic, strong) NSString *authority;
@property (nonatomic, strong) NSString *collection;
@property (nonatomic, strong) NSString *rkey;
- (instancetype)initWithString:(NSString *)uri;
- (NSString *)stringValue;
- (NSString *)description;
@end

@implementation ATURI
- (instancetype)initWithString:(NSString *)uri {
    self = [super init];
    if (self) {
        // Strip "at://" prefix
        NSString *rest = uri;
        if ([rest hasPrefix:@"at://"]) {
            rest = [rest substringFromIndex:5];
        }
        // Split into components
        NSArray *parts = [rest componentsSeparatedByString:@"/"];
        if ([parts count] >= 1) _authority = [parts objectAtIndex:0];
        if ([parts count] >= 2) _collection = [parts objectAtIndex:1];
        if ([parts count] >= 3) _rkey = [parts objectAtIndex:2];
    }
    return self;
}
- (NSString *)stringValue {
    return [NSString stringWithFormat:@"at://%@/%@/%@", _authority, _collection, _rkey];
}
- (NSString *)description {
    return [NSString stringWithFormat:@"ATURI(authority=%@, collection=%@, rkey=%@)", _authority, _collection, _rkey];
}
@end

// Parse some URIs
ATURI *uri1 = [[ATURI alloc] initWithString:@"at://did:plc:abc123/app.bsky.feed.post/3kabc"];
NSLog(@"%@", [uri1 description]);

ATURI *uri2 = [[ATURI alloc] initWithString:@"at://alice.bsky.social/app.bsky.actor.profile/self"];
NSLog(@"%@", [uri2 description]);

// Round-trip
NSLog(@"Round-trip: %@", [uri1 stringValue]);

In [ ]:
@interface RecordStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *records;
- (void)putRecord:(NSDictionary *)record atURI:(NSString *)uri;
- (NSDictionary *)getRecordAtURI:(NSString *)uri;
@end

@implementation RecordStore
- (instancetype)init {
    self = [super init];
    if (self) { _records = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)putRecord:(NSDictionary *)record atURI:(NSString *)uri {
    [_records setObject:record forKey:uri];
    NSLog(@"Stored at %@", uri);
}
- (NSDictionary *)getRecordAtURI:(NSString *)uri {
    NSDictionary *record = [_records objectForKey:uri];
    if (record == nil) {
        NSLog(@"Not found: %@", uri);
    }
    return record;
}
@end

In [ ]:
@interface ATURIResolver : NSObject
@property (nonatomic, strong) RecordStore *store;
- (NSDictionary *)resolveURI:(ATURI *)uri;
- (NSDictionary *)resolveString:(NSString *)uriString;
@end

@implementation ATURIResolver
- (instancetype)initWithStore:(RecordStore *)store {
    self = [super init];
    if (self) { _store = store; }
    return self;
}
- (NSDictionary *)resolveURI:(ATURI *)uri {
    NSString *uriString = [uri stringValue];
    NSDictionary *record = [_store getRecordAtURI:uriString];
    if (record != nil) {
        NSLog(@"Resolved %@ -> %@", [uri description], record);
    }
    return record;
}
- (NSDictionary *)resolveString:(NSString *)uriString {
    ATURI *uri = [[ATURI alloc] initWithString:uriString];
    return [self resolveURI:uri];
}
@end

In [ ]:
RecordStore *store = [[RecordStore alloc] init];
ATURIResolver *resolver = [[ATURIResolver alloc] initWithStore:store];

// Store some records
[store putRecord:@{@"text": @"Hello ATProto!", @"createdAt": @"2025-01-01"}
         atURI:@"at://did:plc:abc123/app.bsky.feed.post/3kabc"];

[store putRecord:@{@"displayName": @"Alice", @"description": @"Test user"}
         atURI:@"at://did:plc:abc123/app.bsky.actor.profile/self"];

// Resolve by constructing ATURI
ATURI *postUri = [[ATURI alloc] initWithString:@"at://did:plc:abc123/app.bsky.feed.post/3kabc"];
NSDictionary *post = [resolver resolveURI:postUri];
NSLog(@"Post text: %@", [post objectForKey:@"text"]);

// Resolve by string
NSDictionary *profile = [resolver resolveString:@"at://did:plc:abc123/app.bsky.actor.profile/self"];
NSLog(@"Profile name: %@", [profile objectForKey:@"displayName"]);

// Non-existent record
NSDictionary *missing = [resolver resolveString:@"at://did:plc:xyz/app.bsky.feed.post/missing"];
NSLog(@"Missing: %@", missing);

## Exercise

Add a `collectionURI` method to `ATURI` that returns just the collection part
(`at://authority/collection`) without the rkey. This is useful for listing all records in a
collection.

Hint: reconstruct the URI without the rkey component.

In [ ]:
// Exercise: implement collectionURI on ATURI
// Your code here...

## Solution

In [ ]:
// Solution: collectionURI returns at://authority/collection
@interface ATURI2 : NSObject
@property (nonatomic, strong) NSString *authority;
@property (nonatomic, strong) NSString *collection;
@property (nonatomic, strong) NSString *rkey;
- (instancetype)initWithString:(NSString *)uri;
- (NSString *)collectionURI;
@end

@implementation ATURI2
- (instancetype)initWithString:(NSString *)uri {
    self = [super init];
    if (self) {
        NSString *rest = uri;
        if ([rest hasPrefix:@"at://"]) rest = [rest substringFromIndex:5];
        NSArray *parts = [rest componentsSeparatedByString:@"/"];
        if ([parts count] >= 1) _authority = [parts objectAtIndex:0];
        if ([parts count] >= 2) _collection = [parts objectAtIndex:1];
        if ([parts count] >= 3) _rkey = [parts objectAtIndex:2];
    }
    return self;
}
- (NSString *)collectionURI {
    return [NSString stringWithFormat:@"at://%@/%@", _authority, _collection];
}
@end

ATURI2 *u = [[ATURI2 alloc] initWithString:@"at://did:plc:abc/app.bsky.feed.post/3k"];
NSLog(@"Collection URI: %@", [u collectionURI]);

## What to Read Next

You now understand all five ATProto identifier types. Next:

- **Part 7: CID Content Addressing** — how content hashes become addresses
- **Part 8: DAG-CBOR** — deterministic binary encoding for ATProto data
- **Part 12: XRPC Dispatch** — how NSIDs route to method handlers